<a href="https://colab.research.google.com/github/BhaskarKumarSinha/Ml-Deep-Learning-AI-Projects/blob/main/GENAIProjects/Text2SQL_via_Prompt_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Text-to-SQL: Bridging the Gap Between Human Language and Databases


Text-to-SQL, also known as Natural Language to SQL (NL2SQL), is a rapidly evolving technology that translates natural, everyday language into Structured Query Language (SQL) commands. This innovative approach empowers users to interact with and retrieve data from databases simply by asking questions in plain English, eliminating the need for specialized knowledge of complex SQL syntax.

At its core, Text-to-SQL acts as an intelligent translator. It leverages the power of artificial intelligence, particularly **Natural Language Processing (NLP)** and sophisticated **AI models**, to understand the user's intent and generate the corresponding SQL query. This process allows individuals without a technical background to explore and analyze data, thereby democratizing data access within an organization.

## How It Works: From a Simple Question to a Complex Query

The conversion of a user's question into an executable SQL query involves a multi-step process:

1.  **Natural Language Understanding (NLU):** The system first analyzes the user's input to decipher its meaning. This involves identifying key entities (like specific columns or tables), the relationships between them, and the user's ultimate goal (e.g., to filter, aggregate, or sort data).

2.  **Schema Linking:** Once the intent is understood, the system maps the identified entities from the natural language question to the specific tables and columns within the database's schema. This is a critical step to ensure the generated query is accurate and relevant to the available data structure.

3.  **SQL Generation:** With the user's intent and the relevant database schema components identified, the AI model constructs the appropriate SQL query. This can range from a simple `SELECT` statement to a complex query involving multiple `JOIN`s, `WHERE` clauses, and aggregate functions.

4.  **Query Execution and Response:** The generated SQL query is then executed against the database. The retrieved data is presented back to the user in a clear and understandable format, often as a table, chart, or a natural language summary.

## Key Applications and Use Cases

The ability to query databases using natural language has a wide array of applications across various industries:

* **Business Intelligence (BI) and Analytics:** Business analysts and decision-makers can quickly get answers to their data-driven questions without relying on data scientists or IT professionals. This accelerates the pace of analysis and reporting.
* **Data Exploration:** For both technical and non-technical users, Text-to-SQL provides an intuitive way to explore large and unfamiliar datasets, uncover insights, and formulate more specific data requests.
* **Customer Support:** Chatbots and virtual assistants integrated with Text-to-SQL can provide customers with real-time information by querying relevant databases based on their questions.
* **E-commerce:** Users can search for products using natural language filters and criteria, which are then translated into database queries to retrieve the most relevant results.

## The Advantages and Challenges

**Benefits:**

* **Increased Accessibility:** It breaks down the barrier to data, allowing a broader range of users to perform data analysis.
* **Improved Efficiency:** It significantly speeds up the process of data retrieval and report generation.
* **Reduced Reliance on Technical Experts:** It frees up data professionals from writing routine queries, allowing them to focus on more complex tasks.

**Challenges:**

* **Ambiguity of Natural Language:** Human language is often imprecise and context-dependent, which can lead to misinterpretation by the AI and the generation of incorrect queries.
* **Complex Database Schemas:** Large and intricately designed databases can pose a significant challenge for the AI to navigate and understand the relationships between numerous tables and columns.
* **Handling Complex Queries:** While proficient at generating simpler queries, Text-to-SQL systems can sometimes struggle with highly complex requests that require deep domain knowledge and intricate logic.

Despite these challenges, the field of Text-to-SQL is continuously advancing, with ongoing research focused on improving the accuracy, robustness, and capabilities of these powerful systems. As AI models become more sophisticated, Text-to-SQL is poised to become an indispensable tool for seamless and intuitive data interaction.

In [ ]:
! curl "https://api.mockaroo.com/api/6d519c10?count=1000&key=8bf84570" > "DepartmentsSchema.csv"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 52412    0 52412    0     0  21712      0 --:--:--  0:00:02 --:--:-- 21720


In [ ]:
! curl "https://api.mockaroo.com/api/a2d22150?count=1000&key=8bf84570" > "EmployeesSchema.csv"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 95605    0 95605    0     0  23808      0 --:--:--  0:00:04 --:--:-- 23811


In [ ]:
! curl "https://api.mockaroo.com/api/ef1d2e00?count=1000&key=8bf84570" > "SalariesSchema.csv"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 80924    0 80924    0     0  23333      0 --:--:--  0:00:03 --:--:-- 23341


## Setup database

In [ ]:
import sqlite3
import pandas as pd
import os

In [ ]:
# Define SQL schemas for creating tables

EmployeesSchema = """
CREATE TABLE IF NOT EXISTS Employees (
    employee_id INTEGER PRIMARY KEY,
    first_name TEXT NOT NULL,
    last_name TEXT NOT NULL,
    age INTEGER,  -- remove CHECK to allow missing ages
    email TEXT UNIQUE NOT NULL,
    gender TEXT,  -- remove CHECK to allow any value
    job_title TEXT NOT NULL,
    department_id INTEGER,
    hire_date DATE,
    FOREIGN KEY (department_id) REFERENCES Departments(department_id)
);
"""

DepartmentsSchema = """
CREATE TABLE IF NOT EXISTS Departments (
    department_id INTEGER PRIMARY KEY,
    department_name TEXT NOT NULL,
    department_manager TEXT,
    department_budget TEXT,  -- change from REAL to TEXT to store currencies like 'Euro'
    department_location TEXT
);
"""

SalariesSchema = """
CREATE TABLE IF NOT EXISTS Salaries (
    salary_id INTEGER PRIMARY KEY AUTOINCREMENT,
    employee_id INTEGER NOT NULL,
    job_title TEXT NOT NULL,
    department TEXT NOT NULL,
    salary TEXT,           -- change to TEXT to allow currency names
    hire_date DATE,
    FOREIGN KEY (employee_id) REFERENCES Employees(employee_id)
);
"""


In [ ]:
db_name = 'departments.db'
if os.path.exists(db_name):
    os.remove(db_name)
    print(f"Removed existing database '{db_name}'.")

In [ ]:
import sqlite3
import pandas as pd
import os

In [ ]:
COLUMN_DATA_TYPES = {
    'Employees': {
        'employee_id': 'Int64',
        'first_name': 'object',
        'last_name': 'object',
        'age': 'Int64',
        'email': 'object',
        'gender': 'object',
        'job_title': 'object',
        'department_id': 'Int64',
        'hire_date': 'datetime64[ns]'
    },
    'Departments': {
        'department_id': 'Int64',
        'department_name': 'object',
        'department_manager': 'object',
        'department_budget': 'object',
        'department_location': 'object'
    },
    'Salaries': {
        'salary_id': 'Int64',
        'employee_id': 'Int64',
        'job_title': 'object',
        'department': 'object',
        'salary': 'object',
        'hire_date': 'datetime64[ns]'
    }
}



In [ ]:
# --- Database setup ---
db_name = 'ecommerce.db'
conn = None  # Initialize connection to None

try:
    # Establish a connection to the SQLite database
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()
    print(f"Database '{db_name}' created and connected successfully. ✅")

    # Create tables
    cursor.execute(EmployeesSchema)
    cursor.execute(DepartmentsSchema)
    cursor.execute(SalariesSchema)
    print("Tables 'Employees', 'Departments', and 'Salaries' created successfully.")

     # 🔥 Drop old tables before recreating them
    cursor.execute("DROP TABLE IF EXISTS Employees;")
    cursor.execute("DROP TABLE IF EXISTS Departments;")
    cursor.execute("DROP TABLE IF EXISTS Salaries;")
    conn.commit()

    # --- Load data from CSV files into the tables using pandas ---
    csv_to_table_map = {
        '/content/DepartmentsSchema.csv': 'Departments',
        '/content/EmployeesSchema.csv': 'Employees',
        '/content/SalariesSchema.csv': 'Salaries'
    }

    for csv_file, table_name in csv_to_table_map.items():
        if os.path.exists(csv_file):
            print(f"\nProcessing '{csv_file}' for table '{table_name}'...")

            # Read the CSV file into a pandas DataFrame
            df = pd.read_csv(csv_file)

            # 1. Get the expected schema for the current table
            expected_schema = COLUMN_DATA_TYPES[table_name]
            expected_cols = list(expected_schema.keys())

            # 2. Handle missing/extra columns
            df = df[df.columns.intersection(expected_cols)]
            for col in expected_cols:
                if col not in df.columns:
                    df[col] = None

            # 3. Reorder columns to match schema
            df = df[expected_cols]

            # 4. Enforce data types
            for col, dtype in expected_schema.items():
                if 'datetime' in dtype:
                    df[col] = pd.to_datetime(df[col], errors='coerce')
                else:
                    try:
                        df[col] = df[col].astype(dtype)
                    except (ValueError, TypeError) as e:
                        print(f"  - Warning: Could not convert column '{col}' to {dtype}. Error: {e}. Leaving as is.")

            # Insert cleaned data
            df.to_sql(table_name, conn, if_exists='append', index=False)
            print(f"  -> Data from '{csv_file}' loaded into '{table_name}' table successfully.")
        else:
            print(f"Warning: '{csv_file}' not found. Skipping data load for '{table_name}'.")

    # Commit the changes to the database
    conn.commit()
    print("\nData committed to the database successfully. 🎉")

except sqlite3.Error as e:
    print(f"Database error: {e}")
except pd.errors.EmptyDataError as e:
    print(f"Pandas error: {e}. One of the CSV files might be empty.")
except KeyError as e:
    print(f"Schema definition error: A column is missing from the TABLE_DATA_TYPES dictionary: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")
finally:
    # Close the connection if it was established
    if conn:
        conn.close()
        print("Database connection closed.")


Database 'ecommerce.db' created and connected successfully. ✅
Tables 'Employees', 'Departments', and 'Salaries' created successfully.

Processing '/content/DepartmentsSchema.csv' for table 'Departments'...
  -> Data from '/content/DepartmentsSchema.csv' loaded into 'Departments' table successfully.

Processing '/content/EmployeesSchema.csv' for table 'Employees'...
  -> Data from '/content/EmployeesSchema.csv' loaded into 'Employees' table successfully.

Processing '/content/SalariesSchema.csv' for table 'Salaries'...
  -> Data from '/content/SalariesSchema.csv' loaded into 'Salaries' table successfully.

Data committed to the database successfully. 🎉
Database connection closed.


##Setup your free API Key using Google's AI Studio

### Import required modules

In [ ]:
from google import genai
from google.colab import userdata

In [ ]:
genai_client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

#Prompt Engineering

## The Anatomy of an Effective Prompt: A Unified Framework

In [ ]:
prompt="""
### ROLE
You are an expert-level SQLite Database Engineer specializing in Natural Language to SQL (NL2SQL) translation. Your only job is to convert plain English queries into accurate SQLite queries.

### CONTEXT
You are the core translation engine for a business intelligence dashboard. This tool allows non-technical employees to query the company's e-commerce database using natural language. The database dialect is always **SQLite**. Your responses will be executed directly on the database.
The database contains the following tables:

**`employees` table:**

```sql
CREATE TABLE employees (
    employee_id INTEGER PRIMARY KEY,
    first_name TEXT NOT NULL,
    last_name TEXT NOT NULL,
    age INTEGER,  -- remove CHECK to allow missing ages
    email TEXT UNIQUE NOT NULL,
    gender TEXT,  -- remove CHECK to allow any value
    job_title TEXT NOT NULL,
    department_id INTEGER,
    hire_date DATE,
    FOREIGN KEY (department_id) REFERENCES Departments(department_id)
);
);
**`departments` table:**

```sql
CREATE TABLE departments (
    department_id INTEGER PRIMARY KEY,
    department_name TEXT NOT NULL,
    department_manager TEXT,
    department_budget TEXT,  -- change from REAL to TEXT to store currencies like 'Euro'
    department_location TEXT
);

**`departments` salaries:**

```sql
CREATE TABLE salaries (
    salary_id INTEGER PRIMARY KEY AUTOINCREMENT,
    employee_id INTEGER NOT NULL,
    job_title TEXT NOT NULL,
    department TEXT NOT NULL,
    salary TEXT,           -- change to TEXT to allow currency names
    hire_date DATE,
    FOREIGN KEY (employee_id) REFERENCES Employees(employee_id)
);

### **TASK**

Your task is to receive a user's question in natural language and convert it into a single, executable SQLite query. Follow these steps meticulously:

1.  **Analyze the User's Query:** Deconstruct the user's question to understand their core intent. Identify the specific data, conditions, aggregations (like `SUM`, `COUNT`, `AVG`), and ordering they are asking for.
2.  **Map to the Schema:** Map the entities from the user's query to the appropriate tables (`employees`, `departments`, `salaries`) and columns. Determine the necessary `JOIN` operations using `employees.employeesr_id` and `salaries.salary_id` as foreign keys in the `orders` table.
3.  **Construct the SQLite Query:** Write a clean and efficient `SELECT` statement that is syntactically correct for SQLite. Ensure all table and column names are accurate.
4.  **Handle Ambiguity:** If the user's query is vague, ambiguous, or lacks the necessary information to create a precise query, do not guess. Instead, formulate a specific, targeted question to ask the user for the missing information.

-----

### **CONSTRAINTS**

  * **Read-Only Operations:** You must **ONLY** generate `SELECT` queries. Never generate `INSERT`, `UPDATE`, `DELETE`, `DROP`, or any other data-modifying statements.
  * **Adhere Strictly to Schema:** Only use the tables and columns defined in the context. Do not invent or assume the existence of any other tables or columns.
  * **No Explanations:** Do not add any conversational text or explanations about the query you generate. Your output must strictly follow the specified format.
  * **Single Query Only:** The final output must be a single, complete, and executable SQL query.
  * **Handle Impossibility:** If a request is impossible to fulfill with the given schema (e.g., "Which employee made the most sales?"), state clearly that the request cannot be completed and briefly explain why.

-----
### **EXAMPLES**

**Example 1: Simple Lookup**

  * **User Query:** "List all employees working in the IT department"
  * **Expected Output:**
    ```json
   {
  "status": "success",
  "response": "SELECT e.* FROM employees e INNER JOIN departments d ON e.department_id = d.department_id WHERE d.department_name = 'IT';"
   }
    ```

**Example 2: Complex Join and Aggregation**

  * **User Query:** User Query: "What is the average salary of employees in each department?"
  * **Expected Output:**
    ```json
  {
    "status": "success",
    "response": "SELECT d.department_name, AVG(s.salary) AS avg_salary FROM employees e INNER JOIN departments d ON e.department_id = d.department_id INNER JOIN salaries s ON e.employee_id = s.employee_id GROUP BY d.department_name;"
  }
    ```

**Example 3: Ambiguous Query**

  * **User Query:** "Show me employees hired recently"
  * **Expected Output:**
    ```json
   {
    "status": "clarification_needed",
    "response": "Could you please define what 'recently' means? For example, 'in the last 30 days', 'this year', or 'since 2020'."
   }
    ```

**Example 4: Impossible Query**

  * **User Query:** Which manager has the most employees?
  * **Expected Output:**
    ```json
   {
    "status": "error",
    "response": "I cannot answer this question as the schema does not contain manager information."
   }
    ```

-----
### **OUTPUT FORMAT**

Your final response must be a single JSON object with two keys:

1.  `"status"`: A string with one of three possible values: `"success"`, `"clarification_needed"`, or `"error"`.
2.  `"response"`:
      * If `status` is `"success"`, this will be a string containing the complete SQLite query.
      * If `status` is `"clarification_needed"`, this will be a string containing the clarifying question for the user.
      * If `status` is `"error"`, this will be a string explaining why the query could not be generated.

"""


In [ ]:
import json
def get_sql_query(genai_client, prompt, user_query):

  # https://www.geeksforgeeks.org/python/formatted-string-literals-f-strings-python/
  contents = f"""
  {prompt}

  Here's the user query in english you need to work on:
  {user_query}
  """
  response = genai_client.models.generate_content(model='gemini-2.5-flash', contents=contents)
  # print(response)

  # Access the usage_metadata attribute
  usage_metadata = response.usage_metadata

  # Print the different token counts
  print(f"Input Token Count: {usage_metadata.prompt_token_count}")
  print(f"Thoughts Token Count: {response.usage_metadata.thoughts_token_count}")
  print(f"Output Token Count: {usage_metadata.candidates_token_count}")
  print(f"Total Token Count: {usage_metadata.total_token_count}")

  output = json.loads(response.text.replace('```json', '').replace('```', ''))

  return output


In [ ]:
import sqlite3
import pandas as pd

def execute_query(query, db_name='ecommerce.db'):

    conn = None
    try:
        # Connect to the database
        conn = sqlite3.connect(db_name)
        cursor = conn.cursor()

        # Execute the query
        print(f"\nExecuting query on '{db_name}':\n{query}")
        cursor.execute(query)

        # Fetch all results
        results = cursor.fetchall()

        # Get column names from the cursor description
        columns = [description[0] for description in cursor.description]

        # Format results as a dataframe for easier use
        results_as_dict = [dict(zip(columns, row)) for row in results]
        results_df = pd.DataFrame(results_as_dict)

        print("Query executed successfully.")
        return results_df

    except sqlite3.Error as e:
        print(f"Database error executing query: {e}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None
    finally:
        if conn:
            conn.close()

In [ ]:
def text2sql(genai_client, prompt, user_query):
  output = get_sql_query(genai_client, prompt, user_query)
  if output['status'] == 'success':
    results = execute_query(output['response'])
    return results
  return output

In [ ]:
text2sql(genai_client, prompt, "Show all employees in the Sales department with salary greater than 20000")

Input Token Count: 1395
Thoughts Token Count: 1015
Output Token Count: 87
Total Token Count: 2497

Executing query on 'ecommerce.db':
SELECT e.* FROM employees e INNER JOIN departments d ON e.department_id = d.department_id INNER JOIN salaries s ON e.employee_id = s.employee_id WHERE d.department_name = 'Sales' AND CAST(s.salary AS REAL) > 20000;
Query executed successfully.


""


In [ ]:
text2sql(genai_client, prompt, "Show me employees hired in last 30 days")

Input Token Count: 1388
Thoughts Token Count: 142
Output Token Count: 41
Total Token Count: 1571

Executing query on 'ecommerce.db':
SELECT * FROM employees WHERE hire_date >= DATE('now', '-30 days');
Query executed successfully.
Empty DataFrame
Columns: []
Index: []


In [ ]:
text2sql(genai_client, prompt, "Show me the top 2 highest salaries")

Input Token Count: 1386
Thoughts Token Count: 117
Output Token Count: 39
Total Token Count: 1542

Executing query on 'ecommerce.db':
SELECT salary FROM salaries ORDER BY CAST(salary AS REAL) DESC LIMIT 2;
Query executed successfully.


,salary
0,Ruble
1,Yuan Renminbi


In [ ]:
text2sql(genai_client, prompt, "Show me the department with the highest average salary")

Input Token Count: 1387
Thoughts Token Count: 484
Output Token Count: 91
Total Token Count: 1962

Executing query on 'ecommerce.db':
SELECT d.department_name, AVG(s.salary) AS average_salary FROM employees e INNER JOIN departments d ON e.department_id = d.department_id INNER JOIN salaries s ON e.employee_id = s.employee_id GROUP BY d.department_name ORDER BY average_salary DESC LIMIT 1;
Query executed successfully.


""
